# Loyers résidentiels à Casablanca — analyse en coupe

Couverture de l'échantillon, loyer médian au m² par quartier et typologie, dispersion,
durées de mise en ligne.

**Règle transversale :** toutes les sorties filtrent sur `qualite >= 2`, excluent les
doublons et les exclusions analytiques (vente, courte durée), et se limitent à la commune
de Casablanca. Une médiane n'est affichée qu'à partir de 30 observations dans la cellule.

Les règles d'exclusion, les biais connus et la spécification de l'indice hédonique sont
dans `METHODO.md`.

In [ ]:
import os, sys, sqlite3
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd()))
import analyse
from db import connect

pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.1f}")
plt.rcParams.update({"figure.figsize": (11, 5), "axes.grid": True, "grid.alpha": 0.3})

DB = os.environ.get("CASA_RENTALS_DB", "data/casa_rentals.db")
conn = connect(DB)
df = analyse.charger(conn)

print(f"base : {DB}")
print(f"{len(df)} annonces valides (qualite >= 2, hors doublons et exclusions)")

## 1. État de la collecte

In [ ]:
profondeur = analyse.profondeur_collecte(conn)
runs = pd.read_sql_query("SELECT * FROM runs ORDER BY id DESC LIMIT 10", conn)

print(f"periode observee : {profondeur['debut']} -> {profondeur['fin']} "
      f"({profondeur['jours']} jours, {profondeur['trimestres']} trimestre(s))")
print(f"annonces en base : {profondeur['n']}")

if df.empty:
    print("\nAucune donnee exploitable : lancer la collecte (python collect.py --probe) "
          "avant d'interpreter la suite. Les cellules ci-dessous s'executent a vide.")
runs

## 2. Couverture

C'est le tableau qui pilote tout le reste : sans effectif suffisant dans une cellule, il
n'y a pas de médiane à publier. Le seuil est de 30 annonces pour une médiane, 80 pour un
intervalle de confiance crédible.

In [ ]:
diag = analyse.diagnostic_couverture(df)
print(f"{diag['observations']} observations | {diag['cellules']} cellules | "
      f"{diag['publiables']} publiables (n>=30) | {diag['avec_ic']} avec IC (n>=80) | "
      f"{diag['sous_seuil']} sous le seuil")

couverture = analyse.table_couverture(df)
couverture

In [ ]:
# Cellules qu'il ne faut pas publier en l'etat, triees par ce qui leur manque.
sous_seuil = analyse.cellules_sous_seuil(df)
print(f"{len(sous_seuil)} cellules sous le seuil de {analyse.SEUIL_MEDIANE} annonces")
sous_seuil.head(40)

In [ ]:
# Lecture visuelle de la couverture : ou l'echantillon est-il reellement exploitable ?
if not couverture.empty:
    table = couverture[analyse.TYPOLOGIES].head(25)
    fig, ax = plt.subplots(figsize=(9, max(4, 0.32 * len(table))))
    im = ax.imshow(table.values, aspect="auto", cmap="YlGnBu")
    ax.set_xticks(range(len(table.columns)), table.columns)
    ax.set_yticks(range(len(table.index)), table.index)
    for i in range(table.shape[0]):
        for j in range(table.shape[1]):
            n = table.values[i, j]
            ax.text(j, i, int(n), ha="center", va="center",
                    color="white" if n > table.values.max() * 0.6 else "black", fontsize=8)
    ax.set_title(f"Effectif par cellule (seuil de publication : {analyse.SEUIL_MEDIANE})")
    ax.grid(False)
    fig.colorbar(im, ax=ax, label="n")
    plt.tight_layout(); plt.show()
else:
    print("pas de donnees a representer")

## 3. Loyer médian au m² par cellule

Les cellules sous le seuil restent visibles avec leur effectif, mais leurs statistiques
sont masquées : on voit qu'elles existent et pourquoi elles sont vides, plutôt que de lire
une médiane calculée sur quatre annonces.

Rappel : ce sont des **loyers demandés**, pas des loyers de transaction. L'écart à la
négociation est typiquement de 5 à 15 % à la baisse.

In [ ]:
cellules = analyse.stats_par_cellule(df)
cellules[cellules["publiable"]] if not cellules.empty else cellules

In [ ]:
# Repli au segment quand les cellules quartier x typologie sont trop fines.
analyse.stats_par_segment(df)

## 4. Dispersion par quartier

In [ ]:
if not df.empty:
    quartiers = (df.groupby("quartier_norm")["loyer_m2"].count()
                   .sort_values(ascending=False).head(15).index.tolist())
    donnees = [df.loc[df["quartier_norm"] == q, "loyer_m2"].dropna().values for q in quartiers]
    effectifs = [len(d) for d in donnees]

    fig, ax = plt.subplots(figsize=(11, 6))
    ax.boxplot(donnees, tick_labels=[f"{q}\n(n={n})" for q, n in zip(quartiers, effectifs)],
               showfliers=False)
    ax.set_ylabel("loyer demande (MAD/m2/mois)")
    ax.set_title("Dispersion du loyer au m2 par quartier — annonces valides")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout(); plt.show()
else:
    print("pas de donnees a representer")

In [ ]:
if not df.empty:
    fig, ax = plt.subplots()
    for t in analyse.TYPOLOGIES:
        serie = df.loc[df["typologie"] == t, "loyer_m2"].dropna()
        if len(serie) > 5:
            ax.hist(serie, bins=30, alpha=0.5, label=f"{t} (n={len(serie)})")
    ax.set_xlabel("loyer demande (MAD/m2/mois)"); ax.set_ylabel("annonces")
    ax.set_title("Distribution du loyer au m2 par typologie")
    ax.legend(); plt.tight_layout(); plt.show()
else:
    print("pas de donnees a representer")

## 5. Durées de mise en ligne — proxy de tension locative

Cet indicateur réagit à la tension avant les prix, mais il n'est pas interprétable tout de
suite : seules les annonces passées à `disparue` ont une durée complète, les annonces
encore actives sont censurées à droite, et tant que la fenêtre d'observation est courte la
médiane est tirée vers le bas. La cellule ci-dessous refuse de conclure avant deux
trimestres de collecte.

Rappel : une annonce qui disparaît peut avoir été louée, retirée, expirée ou republiée
ailleurs. C'est un proxy de tension, pas un délai de location.

In [ ]:
exploitable, message = analyse.durees_exploitables(conn)
print(("OK — " if exploitable else "PREMATURE — ") + message)

durees = analyse.durees_en_ligne(conn, df)
durees if not durees.empty else print("aucune annonce disparue : rien a mesurer.")

In [ ]:
if exploitable and not durees.empty:
    disparues = df[df["statut"] == "disparue"].copy()
    disparues["duree_jours"] = (pd.to_datetime(disparues["last_seen"])
                                - pd.to_datetime(disparues["first_seen"])).dt.days
    fig, ax = plt.subplots()
    ax.hist(disparues["duree_jours"], bins=40)
    ax.set_xlabel("jours en ligne avant disparition"); ax.set_ylabel("annonces")
    ax.set_title(f"Duree de mise en ligne (n={len(disparues)} annonces disparues)")
    plt.tight_layout(); plt.show()
else:
    print("graphique non produit : cf. message ci-dessus.")

## 6. Annexe — version pondérée

Les résultats principaux sont **bruts par cellule**. Cette pondération par le poids
démographique des arrondissements ne figure qu'en annexe, et n'est calculable qu'une fois
`poids_arrondissements.csv` renseigné à la main depuis les données du RGPH. La fonction
refuse de calculer sur des poids partiels plutôt que d'inventer une population.

In [ ]:
try:
    analyse.ponderer_par_arrondissement(df)
except (FileNotFoundError, ValueError) as exc:
    print(f"ponderation non calculee : {exc}")